[Reference](https://medium.com/activated-thinker/build-an-information-retrieval-system-project-from-scratch-d7e13e320c30$0)

# Install KaggleHub

In [1]:
!pip install kagglehub[pandas-datasets]

# Step 1: Load the dataset


In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "gpreda/bbc-news",
    path="bbc_news.csv"
)
df.head()

/tmp/ipykernel_1199/2825993969.py:3: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 12.9M/12.9M [00:00<00:00, 38.4MB/s]


,title,pubDate,guid,link,description
0,Ukraine: Angry Zelensky vows to punish Russian...,"Mon, 07 Mar 2022 08:01:56 GMT",https://www.bbc.co.uk/news/world-europe-60638042,https://www.bbc.co.uk/news/world-europe-606380...,The Ukrainian president says the country will ...
1,War in Ukraine: Taking cover in a town under a...,"Sun, 06 Mar 2022 22:49:58 GMT",https://www.bbc.co.uk/news/world-europe-60641873,https://www.bbc.co.uk/news/world-europe-606418...,"Jeremy Bowen was on the frontline in Irpin, as..."
2,Ukraine war 'catastrophic for global food',"Mon, 07 Mar 2022 00:14:42 GMT",https://www.bbc.co.uk/news/business-60623941,https://www.bbc.co.uk/news/business-60623941?a...,One of the world's biggest fertiliser firms sa...
3,Manchester Arena bombing: Saffie Roussos's par...,"Mon, 07 Mar 2022 00:05:40 GMT",https://www.bbc.co.uk/news/uk-60579079,https://www.bbc.co.uk/news/uk-60579079?at_medi...,The parents of the Manchester Arena bombing's ...
4,Ukraine conflict: Oil price soars to highest l...,"Mon, 07 Mar 2022 08:15:53 GMT",https://www.bbc.co.uk/news/business-60642786,https://www.bbc.co.uk/news/business-60642786?a...,Consumers are feeling the impact of higher ene...


# Step 2: Understand the Dataset


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42115 entries, 0 to 42114
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        42115 non-null  object
 1   pubDate      42115 non-null  object
 2   guid         42115 non-null  object
 3   link         42115 non-null  object
 4   description  42115 non-null  object
dtypes: object(5)
memory usage: 1.6+ MB


# Step 3: Define What a “Document” Is


In [4]:
df["document"] = df["title"] + ". " + df["description"]

In [5]:
documents = df["document"].tolist()
len(documents)
documents[0] # To check how the doc looks like

'Ukraine: Angry Zelensky vows to punish Russian atrocities. The Ukrainian president says the country will not forgive or forget those who murder its civilians.'

# Step 4: BM25 — Keyword-Based Retrieval


## Install BM25 library

In [6]:
!pip install rank-bm25

## Build the BM25 index


In [7]:
from rank_bm25 import BM25Okapi
# Simple tokenization (intentionally basic)
tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

# Run a sample query


In [8]:
query = "uk economy inflation"
tokenized_query = query.lower().split()

In [9]:
scores = bm25.get_scores(tokenized_query)
top_k = 5
top_indices = scores.argsort()[-top_k:][::-1]
for idx in top_indices:
    print("SCORE:", scores[idx])
    print(documents[idx])
    print("-" * 80)

SCORE: 16.035112047908303
Will the UK economy ‘bounce back’ this year?. The Prime Minister calls falling inflation a turning point, is he right?
--------------------------------------------------------------------------------
SCORE: 12.908255754809657
This will be year economy bounces back, Sunak says, after inflation falls. The prime minister tells the BBC the economy has "turned a corner" as he is challenged on rising bills.
--------------------------------------------------------------------------------
SCORE: 12.562818858637126
UK economy shrinks more than expected as rain and strikes hit. Official figures show the UK economy contracted by 0.5% in July, more than economists predicted.
--------------------------------------------------------------------------------
SCORE: 12.416138197205417
Faisal Islam: Can Sunak steer economy through crisis?. The new prime minister must restore market credibility, tackle inflation and drive growth.
-------------------------------------------------

# Step 5: Dense Retrieval — Semantic Search with Embeddings


In [10]:
!pip install sentence-transformers

In [11]:
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
# Load a lightweight embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")
# Create embeddings for all documents
doc_embeddings = model.encode(documents, show_progress_bar=True)
# Create embedding for the query
query = "uk economy inflation"
query_embedding = model.encode([query])[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1317 [00:00<?, ?it/s]

In [ ]:
# Compute similarity using dot product
scores = np.dot(doc_embeddings, query_embedding)

In [ ]:
top_k = 5
top_indices = scores.argsort()[-top_k:][::-1] # descending order
for idx in top_indices:
    print("SCORE:", scores[idx])
    print(documents[idx])
    print("-" * 80)

# Step 6: Hybrid Retrieval — Combining Keywords + Meaning


## Normalize BM25 and dense scores


In [ ]:
# BM25 scores (reuse from earlier)
bm25_scores = bm25.get_scores(tokenized_query)

In [ ]:
# Dense scores (reuse from earlier)
dense_scores = np.dot(doc_embeddings, query_embedding)
# Normalize scores to 0–1 range
bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min())
dense_norm = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min())

## Combine scores (Hybrid Search)

In [ ]:
# Weighted combination
alpha = 0.5  # weight for BM25
hybrid_scores = alpha * bm25_norm + (1 - alpha) * dense_norm

In [ ]:
top_k = 5
top_indices = hybrid_scores.argsort()[-top_k:][::-1]
for idx in top_indices:
    print("SCORE:", hybrid_scores[idx])
    print(documents[idx])
    print("-" * 80)